In [ ]:
# ================================================================
# NBA MASTER DATASET V2 — COMPLETE WINS PREDICTION MODEL
# Model: Tuned Random Forest Regressor
# Target: Regular-season wins
# ================================================================


# ----------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ----------------------------------------------------------------

import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    RepeatedKFold,
    cross_validate,
    train_test_split
)
from sklearn.pipeline import Pipeline


# ----------------------------------------------------------------
# 2. SETTINGS
# ----------------------------------------------------------------

FILE_PATH = "NBA_Master_Dataset_V2_2026.csv"

TARGET = "W"
TEAM_COLUMN = "Team"

TEST_SIZE = 0.25
RANDOM_STATE = 42


# ----------------------------------------------------------------
# 3. LOAD DATASET
# ----------------------------------------------------------------

try:
    df = pd.read_csv(FILE_PATH)

except FileNotFoundError:
    raise FileNotFoundError(
        f"\nThe file '{FILE_PATH}' was not found.\n"
        "Make sure the CSV is in the same folder as your notebook."
    )


print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print(f"File: {FILE_PATH}")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")


# Remove accidental spaces from column names
df.columns = df.columns.str.strip()


# ----------------------------------------------------------------
# 4. CHECK REQUIRED COLUMNS
# ----------------------------------------------------------------

if TARGET not in df.columns:
    raise ValueError(
        f"The target column '{TARGET}' was not found."
    )

if TEAM_COLUMN not in df.columns:
    raise ValueError(
        f"The team column '{TEAM_COLUMN}' was not found."
    )


# ----------------------------------------------------------------
# 5. SELECT FEATURES
# ----------------------------------------------------------------
#
# These features were selected because they represent:
#
# - Financial investment
# - Salary concentration
# - Team age
# - Offensive performance
# - Defensive performance
# - Shooting efficiency
# - Rebounding
# - Passing and turnovers
#
# Features directly calculated from wins were excluded.
#
# Examples excluded:
#
# WinPct
# WinPct_V2
# CostPerWin
# CostPerWin_V2
# WinsPerPayrollMillion
# CapAllocationsPerWin
# DeflectionsPerWin
# LooseBallsRecoveredPerWin
# ContestedShotsPerWin
# TeamPerformanceScore
# TeamPerformanceZ
# FinancialEfficiencyScore
# FrontOfficeCompositeScore
#
# Using those columns would cause target leakage.
# ----------------------------------------------------------------

features = [

    # Financial features
    "PayrollMillions",
    "DeadCashPctOfPayroll",
    "Salary_Top3Share",

    # Roster feature
    "AvgAge",

    # Overall basketball performance
    "ORtg",
    "DRtg",
    "Pace",
    "TS%",

    # Box-score production
    "TRB",
    "AST",
    "TOV",

    # Advanced performance features
    "EffectiveFGDifferential",
    "TurnoverRateDifferential",
    "ReboundingComposite",
    "AssistTurnoverRatio"
]


# Check that every selected feature exists
missing_features = [
    feature
    for feature in features
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        "\nThe following selected features are missing:\n"
        f"{missing_features}"
    )


print("\nSelected model features:")

for number, feature in enumerate(features, start=1):
    print(f"{number}. {feature}")


# ----------------------------------------------------------------
# 6. PREPARE MODELING DATA
# ----------------------------------------------------------------

X = df[features].copy()
y = df[TARGET].copy()


# Convert all model columns to numeric values
for column in X.columns:
    X[column] = pd.to_numeric(
        X[column],
        errors="coerce"
    )

y = pd.to_numeric(
    y,
    errors="coerce"
)


# Remove rows where the target is missing
valid_target_rows = y.notna()

X = X.loc[valid_target_rows].copy()
y = y.loc[valid_target_rows].copy()

model_df = df.loc[valid_target_rows].copy()


print("\nMissing values before imputation:")
print(X.isna().sum()[X.isna().sum() > 0])


# ----------------------------------------------------------------
# 7. TRAIN/TEST SPLIT
# ----------------------------------------------------------------
#
# 75% training data
# 25% testing data
#
# Because there are 30 teams:
#
# Approximately 22 teams are used for training.
# Approximately 8 teams are used for testing.
# ----------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)


print("\n" + "=" * 70)
print("TRAIN/TEST SPLIT")
print("=" * 70)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")


# ----------------------------------------------------------------
# 8. CREATE MODEL PIPELINE
# ----------------------------------------------------------------
#
# The pipeline:
#
# 1. Replaces missing values with the median
# 2. Trains the Random Forest
#
# Putting preprocessing inside the pipeline prevents information
# from the testing set from affecting the training process.
# ----------------------------------------------------------------

pipeline = Pipeline(

    steps=[

        (
            "imputer",
            SimpleImputer(strategy="median")
        ),

        (
            "model",
            RandomForestRegressor(
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)


# ----------------------------------------------------------------
# 9. HYPERPARAMETER GRID
# ----------------------------------------------------------------
#
# GridSearchCV will test different Random Forest settings and
# select the combination with the lowest cross-validation MAE.
# ----------------------------------------------------------------

parameter_grid = {

    "model__n_estimators": [
        300,
        600
    ],

    "model__max_depth": [
        3,
        5,
        None
    ],

    "model__min_samples_split": [
        2,
        4
    ],

    "model__min_samples_leaf": [
        1,
        2
    ],

    "model__max_features": [
        "sqrt",
        0.7
    ]
}


# ----------------------------------------------------------------
# 10. FIVE-FOLD CROSS-VALIDATION SETUP
# ----------------------------------------------------------------

grid_cv = KFold(

    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ----------------------------------------------------------------
# 11. TUNE THE RANDOM FOREST
# ----------------------------------------------------------------
#
# Negative MAE is used instead of R² during tuning because each
# fold contains only a few NBA teams. MAE is more stable and easier
# to interpret with a small dataset.
# ----------------------------------------------------------------

grid_search = GridSearchCV(

    estimator=pipeline,

    param_grid=parameter_grid,

    scoring="neg_mean_absolute_error",

    cv=grid_cv,

    n_jobs=-1,

    return_train_score=True
)


grid_search.fit(
    X_train,
    y_train
)


best_model = grid_search.best_estimator_


print("\n" + "=" * 70)
print("BEST RANDOM FOREST SETTINGS")
print("=" * 70)

for parameter, value in grid_search.best_params_.items():

    clean_parameter = parameter.replace(
        "model__",
        ""
    )

    print(f"{clean_parameter}: {value}")


best_grid_mae = -grid_search.best_score_

print(
    f"\nBest training cross-validation MAE: "
    f"{best_grid_mae:.2f} wins"
)


# ----------------------------------------------------------------
# 12. TEST-SET PREDICTIONS
# ----------------------------------------------------------------

test_predictions = best_model.predict(
    X_test
)


# NBA wins should be between 0 and 82
test_predictions = np.clip(
    test_predictions,
    0,
    82
)


# ----------------------------------------------------------------
# 13. FINAL TEST-SET EVALUATION
# ----------------------------------------------------------------

test_mae = mean_absolute_error(
    y_test,
    test_predictions
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        test_predictions
    )
)

test_mape = mean_absolute_percentage_error(
    y_test,
    test_predictions
)

test_r2 = r2_score(
    y_test,
    test_predictions
)


print("\n" + "=" * 70)
print("FINAL TEST-SET PERFORMANCE")
print("=" * 70)

print(f"MAE:  {test_mae:.2f} wins")
print(f"RMSE: {test_rmse:.2f} wins")
print(f"MAPE: {test_mape:.2%}")
print(f"R²:   {test_r2:.3f}")


print("\nMetric explanations:")

print(
    f"The model's predictions are off by approximately "
    f"{test_mae:.2f} wins on average."
)

print(
    f"The RMSE is {test_rmse:.2f}, which gives extra weight "
    "to larger prediction errors."
)

print(
    f"The R² score means the model explains approximately "
    f"{test_r2 * 100:.1f}% of the variation in test-set wins."
)


# ----------------------------------------------------------------
# 14. REPEATED CROSS-VALIDATION
# ----------------------------------------------------------------
#
# A single test split can be unstable with only 30 teams.
#
# Repeated five-fold cross-validation tests the model across
# multiple different data splits and gives a more reliable
# estimate of model performance.
# ----------------------------------------------------------------

repeated_cv = RepeatedKFold(

    n_splits=5,
    n_repeats=10,
    random_state=RANDOM_STATE
)


cv_results = cross_validate(

    estimator=best_model,

    X=X,
    y=y,

    cv=repeated_cv,

    scoring={

        "mae":
        "neg_mean_absolute_error",

        "rmse":
        "neg_root_mean_squared_error",

        "r2":
        "r2"
    },

    n_jobs=-1
)


cv_mae_scores = -cv_results["test_mae"]
cv_rmse_scores = -cv_results["test_rmse"]
cv_r2_scores = cv_results["test_r2"]


print("\n" + "=" * 70)
print("REPEATED CROSS-VALIDATION RESULTS")
print("=" * 70)

print(
    f"Average CV MAE: "
    f"{cv_mae_scores.mean():.2f} wins"
)

print(
    f"CV MAE standard deviation: "
    f"{cv_mae_scores.std():.2f}"
)

print(
    f"Average CV RMSE: "
    f"{cv_rmse_scores.mean():.2f} wins"
)

print(
    f"Average CV R²: "
    f"{cv_r2_scores.mean():.3f}"
)

print(
    f"CV R² standard deviation: "
    f"{cv_r2_scores.std():.3f}"
)


# ----------------------------------------------------------------
# 15. CREATE TEST PREDICTION TABLE
# ----------------------------------------------------------------

test_results = pd.DataFrame({

    "Team":
    model_df.loc[y_test.index, TEAM_COLUMN],

    "ActualWins":
    y_test,

    "PredictedWins":
    test_predictions
})


test_results["PredictionError"] = (

    test_results["PredictedWins"]
    -
    test_results["ActualWins"]
)


test_results["AbsoluteError"] = (

    test_results["PredictionError"]
    .abs()
)


test_results["PredictionDirection"] = np.where(

    test_results["PredictionError"] > 0,

    "Overpredicted",

    np.where(

        test_results["PredictionError"] < 0,

        "Underpredicted",

        "Exact"
    )
)


test_results["PredictedWins"] = (
    test_results["PredictedWins"]
    .round(1)
)

test_results["PredictionError"] = (
    test_results["PredictionError"]
    .round(1)
)

test_results["AbsoluteError"] = (
    test_results["AbsoluteError"]
    .round(1)
)


test_results = test_results.sort_values(

    by="AbsoluteError",

    ascending=True
)


print("\n" + "=" * 70)
print("TEST-SET TEAM PREDICTIONS")
print("=" * 70)

print(
    test_results.to_string(
        index=False
    )
)


# ----------------------------------------------------------------
# 16. PERMUTATION FEATURE IMPORTANCE
# ----------------------------------------------------------------
#
# Permutation importance measures how much model performance
# declines when each feature is randomly rearranged.
#
# This is generally more trustworthy than relying only on the
# Random Forest's built-in feature importance.
# ----------------------------------------------------------------

permutation_results = permutation_importance(

    estimator=best_model,

    X=X_test,

    y=y_test,

    scoring="neg_mean_absolute_error",

    n_repeats=30,

    random_state=RANDOM_STATE,

    n_jobs=-1
)


importance_df = pd.DataFrame({

    "Feature":
    features,

    "Importance":
    permutation_results.importances_mean,

    "ImportanceStandardDeviation":
    permutation_results.importances_std
})


importance_df = importance_df.sort_values(

    by="Importance",

    ascending=False

).reset_index(drop=True)


print("\n" + "=" * 70)
print("PERMUTATION FEATURE IMPORTANCE")
print("=" * 70)

print(
    importance_df.round(4).to_string(
        index=False
    )
)


# ----------------------------------------------------------------
# 17. ACTUAL VS. PREDICTED GRAPH
# ----------------------------------------------------------------

plt.figure(figsize=(8, 6))

plt.scatter(

    y_test,

    test_predictions,

    s=100
)


minimum_value = min(
    y_test.min(),
    test_predictions.min()
)

maximum_value = max(
    y_test.max(),
    test_predictions.max()
)


plt.plot(

    [minimum_value, maximum_value],

    [minimum_value, maximum_value],

    linestyle="--"
)


for index in y_test.index:

    team_name = model_df.loc[
        index,
        TEAM_COLUMN
    ]

    actual_value = y_test.loc[index]

    prediction_position = list(
        y_test.index
    ).index(index)

    predicted_value = test_predictions[
        prediction_position
    ]

    plt.annotate(

        team_name,

        (
            actual_value,
            predicted_value
        ),

        xytext=(5, 5),

        textcoords="offset points",

        fontsize=8
    )


plt.xlabel("Actual Wins")
plt.ylabel("Predicted Wins")

plt.title(
    "NBA Wins Model: Actual vs. Predicted Wins"
)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# ----------------------------------------------------------------
# 18. RESIDUAL GRAPH
# ----------------------------------------------------------------
#
# Residual:
#
# Actual wins - Predicted wins
#
# Positive residual:
# Model underpredicted the team.
#
# Negative residual:
# Model overpredicted the team.
# ----------------------------------------------------------------

residuals = (
    y_test.values
    -
    test_predictions
)


plt.figure(figsize=(8, 6))

plt.scatter(

    test_predictions,

    residuals,

    s=100
)


plt.axhline(

    y=0,

    linestyle="--"
)


for index, residual in zip(
    y_test.index,
    residuals
):

    team_name = model_df.loc[
        index,
        TEAM_COLUMN
    ]

    prediction_position = list(
        y_test.index
    ).index(index)

    predicted_value = test_predictions[
        prediction_position
    ]

    plt.annotate(

        team_name,

        (
            predicted_value,
            residual
        ),

        xytext=(5, 5),

        textcoords="offset points",

        fontsize=8
    )


plt.xlabel("Predicted Wins")
plt.ylabel("Residual: Actual Wins - Predicted Wins")

plt.title(
    "NBA Wins Model Residual Plot"
)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# ----------------------------------------------------------------
# 19. TOP FEATURE IMPORTANCE GRAPH
# ----------------------------------------------------------------

top_features = importance_df.head(10).sort_values(

    by="Importance",

    ascending=True
)


plt.figure(figsize=(10, 6))

plt.barh(

    top_features["Feature"],

    top_features["Importance"]
)


plt.xlabel(
    "Permutation Importance"
)

plt.ylabel(
    "Feature"
)

plt.title(
    "Top 10 Features for Predicting NBA Wins"
)

plt.tight_layout()
plt.show()


# ----------------------------------------------------------------
# 20. TRAIN FINAL MODEL ON ALL 30 TEAMS
# ----------------------------------------------------------------
#
# The held-out test set was used for honest evaluation.
#
# After evaluation is complete, the best model is retrained using
# all available teams so it can use the maximum amount of data
# when being saved or used inside the dashboard.
# ----------------------------------------------------------------

final_model = best_model

final_model.fit(
    X,
    y
)


# ----------------------------------------------------------------
# 21. PREDICT ALL TEAMS
# ----------------------------------------------------------------

full_predictions = final_model.predict(
    X
)


full_predictions = np.clip(

    full_predictions,

    0,
    82
)


prediction_output = model_df.copy()


prediction_output[
    "ModelPredictedWins"
] = np.round(

    full_predictions,

    1
)


prediction_output[
    "ModelPredictionError"
] = np.round(

    prediction_output[
        "ModelPredictedWins"
    ]

    -

    prediction_output[
        TARGET
    ],

    1
)


prediction_output[
    "ModelAbsoluteError"
] = np.round(

    prediction_output[
        "ModelPredictionError"
    ].abs(),

    1
)


league_predictions = prediction_output[[

    TEAM_COLUMN,

    TARGET,

    "ModelPredictedWins",

    "ModelPredictionError",

    "ModelAbsoluteError"

]].copy()


league_predictions = league_predictions.rename(

    columns={

        TARGET:
        "ActualWins"
    }
)


league_predictions = league_predictions.sort_values(

    by="ModelPredictedWins",

    ascending=False
)


print("\n" + "=" * 70)
print("FULL LEAGUE MODEL PREDICTIONS")
print("=" * 70)

print(

    league_predictions.to_string(

        index=False
    )
)


# ----------------------------------------------------------------
# 22. EXPORT RESULTS
# ----------------------------------------------------------------

prediction_output.to_csv(

    "NBA_Master_Dataset_With_Predictions.csv",

    index=False
)


league_predictions.to_csv(

    "NBA_Wins_Predictions.csv",

    index=False
)


test_results.to_csv(

    "NBA_Test_Set_Predictions.csv",

    index=False
)


importance_df.to_csv(

    "NBA_Model_Feature_Importance.csv",

    index=False
)


# ----------------------------------------------------------------
# 23. SAVE THE TRAINED MODEL
# ----------------------------------------------------------------
#
# Save the model together with the feature list and other
# information needed to use it later.
# ----------------------------------------------------------------

model_package = {

    "model":
    final_model,

    "features":
    features,

    "target":
    TARGET,

    "team_column":
    TEAM_COLUMN,

    "best_parameters":
    grid_search.best_params_,

    "test_mae":
    test_mae,

    "test_rmse":
    test_rmse,

    "test_mape":
    test_mape,

    "test_r2":
    test_r2,

    "cross_validation_mae":
    cv_mae_scores.mean(),

    "cross_validation_rmse":
    cv_rmse_scores.mean(),

    "cross_validation_r2":
    cv_r2_scores.mean()
}


joblib.dump(

    model_package,

    "NBA_Wins_Model_Package.pkl"
)


# ----------------------------------------------------------------
# 24. FINAL SUMMARY
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL COMPLETE")
print("=" * 70)

print(
    "\nFiles created:"
)

print(
    "1. NBA_Master_Dataset_With_Predictions.csv"
)

print(
    "2. NBA_Wins_Predictions.csv"
)

print(
    "3. NBA_Test_Set_Predictions.csv"
)

print(
    "4. NBA_Model_Feature_Importance.csv"
)

print(
    "5. NBA_Wins_Model_Package.pkl"
)

print(
    "\nFinal test performance:"
)

print(
    f"MAE: {test_mae:.2f} wins"
)

print(
    f"RMSE: {test_rmse:.2f} wins"
)

print(
    f"MAPE: {test_mape:.2%}"
)

print(
    f"R²: {test_r2:.3f}"
)

print(
    "\nRepeated cross-validation performance:"
)

print(
    f"Average MAE: "
    f"{cv_mae_scores.mean():.2f} wins"
)

print(
    f"Average RMSE: "
    f"{cv_rmse_scores.mean():.2f} wins"
)

print(
    f"Average R²: "
    f"{cv_r2_scores.mean():.3f}"
)

print(
    "\nThe model and all supporting files were saved successfully."
)